# Nanoparticle Detection Model — Training Pipeline
## Data preparation, label conversion, and YOLO model training for nanoparticle detection in microscopy videos

**Author:** Ben  
**Institution:** Durham University  
**Last Updated:** March 2026

---

### Pipeline Overview
This notebook prepares manually labelled microscopy data for YOLO training and runs the model training process.

### Steps
1. Label Conversion — Convert Fiji/ImageJ CSV labels to YOLO format
2. Dataset Preparation — Tile frames and labels, split into train/val sets
3. Model Training — Fine-tune YOLOv8 on prepared dataset

### Prerequisites
- Manually labelled particle coordinates exported as CSV from Fiji/ImageJ
- Raw TIF frame sequences for each labelled video
- Google Colab with GPU runtime enabled (Runtime → Change runtime type → GPU)

In [ ]:
# =============================================================================
# ENVIRONMENT SETUP
# =============================================================================
# Run once to install required dependencies.
# Requires GPU runtime in Google Colab for training.
# =============================================================================

!pip install ultralytics==8.4.6

In [ ]:
# =============================================================================
# GOOGLE DRIVE MOUNT — COLAB ONLY
# =============================================================================
# Only required when running in Google Colab.
# Skip this cell if running locally in Jupyter.
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
# Standard library
import os
import glob
import shutil
import random
import time
from collections import defaultdict

# Third-party — data handling
import numpy as np
import pandas as pd
import yaml

# Third-party — image processing
from PIL import Image
import cv2

# Third-party — deep learning
from ultralytics import YOLO

# Third-party — progress bars
from tqdm import tqdm

# Type hints
from typing import List, Dict, Tuple, Optional

print("All libraries imported successfully.")

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
# This is the only cell that needs to be edited for each training run.
# =============================================================================

# -----------------------------------------------------------------------------
# GOOGLE DRIVE PATHS — persistent storage
# -----------------------------------------------------------------------------
MASTER_INPUT_ROOT = "/path/to/labelled/video/folders" # Labelled Video Folders
MODEL_OUTPUT_DIR  = "/path/to/model/output"           # Trained model saved here

# -----------------------------------------------------------------------------
# LOCAL COLAB PATHS — fast intermediate storage (lost when runtime disconnects)
# -----------------------------------------------------------------------------
YOLO_OUTPUT_ROOT  = "/content/yolo_labels"    # Step 1 output — converted YOLO labels
TILE_OUTPUT_ROOT  = "/content/yolo_tiles"     # Step 2 output — tiled frames and labels
DATASET_ROOT      = "/content/yolo_dataset"   # Step 3 output — train/val/test split

# -----------------------------------------------------------------------------
# MICROSCOPY SETTINGS
# -----------------------------------------------------------------------------
# Critical: Defines the physical-to-digital conversion ratio.
# Must be updated if microscope magnification changes to ensure spatial accuracy.
PIXELS_PER_MICRON = 9.2308

# -----------------------------------------------------------------------------
# TILING SETTINGS
# -----------------------------------------------------------------------------
TILE_SIZE     = 640   # Tile size in pixels — must match inference pipeline (default 640)
NUM_TILES     = 5     # Number of horizontal tiles per frame
MIN_VISIBILITY = 0.30  # Minimum visibility ratio to retain a label (0.30 = 30%)

# -----------------------------------------------------------------------------
# DATASET SPLIT SETTINGS
# -----------------------------------------------------------------------------
VALIDATION_RATIO = 0.20   # 20% of frames used for validation
TEST_RATIO       = 0.10   # 10% of frames used for testing
RANDOM_SEED      = 42     # Fixed seed for reproducible splits

# -----------------------------------------------------------------------------
# TRAINING SETTINGS
# -----------------------------------------------------------------------------
BASE_MODEL   = "yolov8m.pt"          # Base YOLO model to fine-tune from
EPOCHS       = 50                     # Number of training epochs
BATCH_SIZE   = 16                     # Batch size — reduce if running out of GPU memory
IMAGE_SIZE   = 640                    # Training image size in pixels
RUN_NAME     = "training_run"         # Name for this training run — change each run

In [ ]:
# =============================================================================
# STEP 1 — LABEL CONVERSION
# =============================================================================
# Converts Fiji/ImageJ CSV labels (microns) to YOLO format (normalised 0-1).
# Each CSV frame label is converted to a corresponding .txt file.
# Empty .txt files are created for frames with no particles — this is required
# by YOLO to correctly match frame counts during training.
# =============================================================================

def get_image_dimensions(image_path: str) -> Tuple[Optional[int], Optional[int]]:
    """
    Read width and height from a single image frame.

    Parameters:
        image_path (str): Full path to the reference .tif image.

    Returns:
        Tuple[int, int]: (width, height) in pixels. Returns (None, None) on failure.
    """
    try:
        with Image.open(image_path) as img:
            return img.width, img.height
    except FileNotFoundError:
        print(f"Error: Image not found at {image_path}")
        return None, None
    except Exception as e:
        print(f"Error reading image {image_path}: {e}")
        return None, None


def convert_fiji_csv_to_yolo(
    csv_path: str,
    img_width: int,
    img_height: int
) -> List[str]:
    """
    Convert a Fiji/ImageJ CSV label file to YOLO format.

    Reads bounding box coordinates in microns (BX, BY, Width, Height),
    converts to pixels, calculates centres, and normalises by image dimensions.

    Parameters:
        csv_path (str): Path to the source Fiji CSV file.
        img_width (int): Width of the video frame in pixels.
        img_height (int): Height of the video frame in pixels.

    Returns:
        List[str]: YOLO formatted lines — "class_id x_centre y_centre width height".
                   Returns empty list if no valid particles found.
    """
    try:
        df = pd.read_csv(csv_path)

        # Assign class_id=0 universally — current pipeline tracks a single particle
        # class. Future multi-class tracking will require modification here.
        df['class_id'] = 0
        df.dropna(subset=['BX'], inplace=True)

        if df.empty:
            return []

        # --- Unit Conversion (microns to pixels) ---
        bx_px     = df['BX']     * PIXELS_PER_MICRON
        by_px     = df['BY']     * PIXELS_PER_MICRON
        width_px  = df['Width']  * PIXELS_PER_MICRON
        height_px = df['Height'] * PIXELS_PER_MICRON

        # --- Centroid Calculation ---
        # Fiji provides top-left coordinates (BX, BY).
        # YOLO requires centre coordinates (cx, cy).
        center_x_px = bx_px + (width_px / 2)
        center_y_px = by_px + (height_px / 2)

        # --- Normalisation (0.0 to 1.0) ---
        x_norm = center_x_px / img_width
        y_norm = center_y_px / img_height
        w_norm = width_px    / img_width
        h_norm = height_px   / img_height

        # --- Compile Output Lines ---
        yolo_lines = []
        for i in range(len(df)):
            line = (
                f"{df['class_id'].iloc[i]} "
                f"{x_norm.iloc[i]:.6f} "
                f"{y_norm.iloc[i]:.6f} "
                f"{w_norm.iloc[i]:.6f} "
                f"{h_norm.iloc[i]:.6f}"
            )
            yolo_lines.append(line)

        return yolo_lines

    except Exception as e:
        print(f"Error processing {os.path.basename(csv_path)}: {e}")
        return []


# --- MAIN EXECUTION ---
os.makedirs(YOLO_OUTPUT_ROOT, exist_ok=True)

movie_paths = glob.glob(os.path.join(MASTER_INPUT_ROOT, '*'))
print(f"Found {len(movie_paths)} items in input directory. Starting label conversion...")

files_processed_count = 0

for movie_folder in movie_paths:
    if not os.path.isdir(movie_folder):
        continue

    movie_name  = os.path.basename(movie_folder)
    frames_dir  = os.path.join(movie_folder, 'image_sequence')
    labels_dir  = os.path.join(movie_folder, 'frame_csv_labels')

    # Validation — skip folder if missing frames or labels
    sample_images = glob.glob(os.path.join(frames_dir, '*.tif'))
    if not sample_images:
        continue

    img_width, img_height = get_image_dimensions(sample_images[0])
    if img_width is None:
        continue

    if not os.path.isdir(labels_dir):
        continue

    input_csvs = glob.glob(os.path.join(labels_dir, '*_raw_boxes.csv'))
    if not input_csvs:
        continue

    for csv_file in input_csvs:
        yolo_lines = convert_fiji_csv_to_yolo(csv_file, img_width, img_height)

        # Generate output filename — "frame_001_raw_boxes.csv" -> "frame_001.txt"
        base_name       = os.path.basename(csv_file).replace('_raw_boxes.csv', '')
        output_filename = f"{base_name}.txt"
        save_path       = os.path.join(YOLO_OUTPUT_ROOT, output_filename)

        # Write label file — empty file created if no particles found (YOLO standard)
        with open(save_path, 'w') as f:
            if yolo_lines:
                f.write('\n'.join(yolo_lines))

        files_processed_count += 1

        if files_processed_count % 500 == 0:
            print(f"   Processed {files_processed_count} files (last: {output_filename})")

print(f"Label conversion complete. {files_processed_count} label files generated.")
print(f"Saved to: {YOLO_OUTPUT_ROOT}")

In [ ]:
# =============================================================================
# STEP 2 — FRAME TILING
# =============================================================================
# Tiles preprocessed video frames and their corresponding YOLO labels into
# 640x640 patches for model training. Each frame is padded to 640px height
# then divided into NUM_TILES overlapping horizontal tiles. Labels are
# filtered by visibility — boxes must be at least MIN_VISIBILITY visible
# within a tile to be retained.
# =============================================================================

def load_yolo_labels(label_path: str) -> np.ndarray:
    """
    Load a YOLO .txt label file into a numpy array.
    Handles empty files safely by returning an empty array.

    Parameters:
        label_path (str): Full path to the YOLO .txt label file.

    Returns:
        np.ndarray: Array of shape (N, 5) with columns [class, x, y, w, h].
                    Returns empty (0, 5) array if file is empty or missing.
    """
    if not os.path.exists(label_path):
        raise FileNotFoundError(f"Missing label file at {label_path}")

    if os.path.getsize(label_path) == 0:
        return np.zeros((0, 5), dtype=np.float32)

    try:
        labels = np.loadtxt(label_path, dtype=np.float32)
    except Exception:
        return np.zeros((0, 5), dtype=np.float32)

    if labels.ndim == 1:
        labels = np.expand_dims(labels, axis=0)

    return labels


def pad_image_and_labels(
    image: np.ndarray,
    labels: np.ndarray,
    target_dim: int
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Pad image height to target dimension and adjust YOLO labels accordingly.
    Black bars are added top and bottom to reach target_dim height.

    Parameters:
        image (np.ndarray): Input frame.
        labels (np.ndarray): YOLO labels array of shape (N, 5).
        target_dim (int): Target height in pixels.

    Returns:
        Tuple[np.ndarray, np.ndarray]: (padded image, adjusted labels).
    """
    frame_height, frame_width = image.shape[:2]
    h_pad_needed = target_dim - frame_height

    if h_pad_needed < 0:
        return image, labels

    top_pad    = h_pad_needed // 2
    bottom_pad = h_pad_needed - top_pad

    padded_image = cv2.copyMakeBorder(
        image, top_pad, bottom_pad, 0, 0, cv2.BORDER_CONSTANT, value=0
    )

    if labels.shape[0] > 0:
        # Denormalise, shift, and renormalise y-centre to account for padding
        labels[:, 2] = labels[:, 2] * frame_height
        labels[:, 2] = labels[:, 2] + top_pad
        labels[:, 2] = labels[:, 2] / target_dim
        # Rescale height relative to new padded dimension
        labels[:, 4] = labels[:, 4] * (frame_height / target_dim)

    return padded_image, labels


def calculate_tile_starts(
    frame_width: int,
    tile_width: int,
    num_tiles: int
) -> List[int]:
    """
    Calculate evenly distributed tile start positions along the x-axis.

    Parameters:
        frame_width (int): Width of the full frame in pixels.
        tile_width (int): Width of each tile in pixels.
        num_tiles (int): Number of tiles to generate.

    Returns:
        List[int]: X-axis start positions for each tile.
    """
    total_advance = frame_width - tile_width

    if total_advance <= 0 or num_tiles <= 1:
        return [0]

    num_steps = num_tiles - 1
    base_step = total_advance // num_steps
    remainder = total_advance % num_steps

    starts = [0]
    current_start = 0
    for tile_idx in range(1, num_tiles):
        step = base_step + (1 if tile_idx <= remainder else 0)
        current_start += step
        starts.append(current_start)

    return starts


def create_tile(
    base_name: str,
    tile_idx: int,
    start_x: int,
    tile_dim: int,
    padded_image: np.ndarray,
    labels: np.ndarray,
    output_dir: str
) -> None:
    """
    Extract a single tile from a padded frame, filter labels by visibility,
    and save the tile image and label file to disk.

    Parameters:
        base_name (str): Base filename for output files.
        tile_idx (int): Index of this tile (used in filename).
        start_x (int): X-axis start position of this tile in pixels.
        tile_dim (int): Tile size in pixels.
        padded_image (np.ndarray): Full padded frame.
        labels (np.ndarray): YOLO labels array of shape (N, 5).
        output_dir (str): Directory to save tile image and label file.
    """
    end_x         = start_x + tile_dim
    frame_height  = padded_image.shape[0]
    padded_width  = padded_image.shape[1]

    # Crop tile from padded frame
    tile_image = padded_image[0:frame_height, start_x:end_x]

    valid_tile_labels = []

    if labels.shape[0] > 0:
        # Convert normalised coordinates to pixels
        global_cx = labels[:, 1] * padded_width
        global_cy = labels[:, 2] * frame_height
        global_w  = labels[:, 3] * padded_width
        global_h  = labels[:, 4] * frame_height

        global_left  = global_cx - (global_w / 2)
        global_right = global_cx + (global_w / 2)

        # Calculate how much of each box is visible within this tile
        inner_left  = np.maximum(global_left,  start_x)
        inner_right = np.minimum(global_right, end_x)
        visible_w        = inner_right - inner_left
        visibility_ratio = visible_w / global_w

        # Keep only boxes that meet the minimum visibility threshold
        keep_mask = (visible_w > 0) & (visibility_ratio > MIN_VISIBILITY)

        if np.any(keep_mask):
            new_pixel_w  = inner_right[keep_mask] - inner_left[keep_mask]
            new_pixel_cx = inner_left[keep_mask] + (new_pixel_w / 2)

            final_x = (new_pixel_cx - start_x) / tile_dim
            final_w = new_pixel_w / tile_dim
            final_y = global_cy[keep_mask] / frame_height
            final_h = global_h[keep_mask]  / frame_height

            valid_tile_labels = np.column_stack((
                labels[keep_mask, 0], final_x, final_y, final_w, final_h
            ))

    # Save tile image and label file
    filename_base = f"{base_name}_tile_{tile_idx + 1:02d}_x{start_x}"
    img_path      = os.path.join(output_dir, f"{filename_base}.tif")
    txt_path      = os.path.join(output_dir, f"{filename_base}.txt")

    cv2.imwrite(img_path, tile_image)

    if len(valid_tile_labels) > 0:
        valid_tile_labels[:, 1:] = np.clip(valid_tile_labels[:, 1:], 0.000001, 0.999999)
        np.savetxt(txt_path, valid_tile_labels, fmt=['%d', '%.6f', '%.6f', '%.6f', '%.6f'])
    else:
        open(txt_path, 'w').close()


# --- MAIN EXECUTION ---

if os.path.exists(TILE_OUTPUT_ROOT):
    print(f"Clearing existing output directory: {TILE_OUTPUT_ROOT}")
    shutil.rmtree(TILE_OUTPUT_ROOT)
    time.sleep(1.0)

os.makedirs(TILE_OUTPUT_ROOT, exist_ok=True)
print(f"Output directory ready: {TILE_OUTPUT_ROOT}")

label_files = glob.glob(os.path.join(YOLO_OUTPUT_ROOT, "*.txt"))
print(f"Found {len(label_files)} label files. Starting tiling...")

tiles_created_count = 0

for label_path in label_files:
    base_name = os.path.splitext(os.path.basename(label_path))[0]

    # Find matching thresholded frame images
    search_pattern = os.path.join(
        MASTER_INPUT_ROOT, '**', 'thresholded_frames', f"{base_name}_tv*.tif"
    )
    matching_images = glob.glob(search_pattern, recursive=True)

    if not matching_images:
        continue

    labels = load_yolo_labels(label_path)

    for image_path in matching_images:
        image = cv2.imread(image_path)
        if image is None:
            continue

        padded_image, padded_labels = pad_image_and_labels(
            image, labels, TILE_SIZE
        )
        tile_starts = calculate_tile_starts(
            padded_image.shape[1], TILE_SIZE, NUM_TILES
        )

        for tile_idx, start_x in enumerate(tile_starts):
            tv_base_name = os.path.splitext(os.path.basename(image_path))[0]
            create_tile(
                base_name  = tv_base_name,
                tile_idx   = tile_idx,
                start_x    = start_x,
                tile_dim   = TILE_SIZE,
                padded_image = padded_image,
                labels     = padded_labels,
                output_dir = TILE_OUTPUT_ROOT
            )
            tiles_created_count += 1

    if tiles_created_count % 1000 == 0 and tiles_created_count > 0:
        print(f"   Generated {tiles_created_count} tiles so far...")

print(f"Tiling complete. {tiles_created_count} tiles generated.")
print(f"Saved to: {TILE_OUTPUT_ROOT}")

In [ ]:
# =============================================================================
# STEP 3 — DATASET SPLITTING
# =============================================================================
# Splits tiled frames into train/val/test sets for YOLO training.
# Files are grouped by parent frame ID before splitting to prevent data
# leakage — all tiles from the same original frame are always kept together
# in the same split. This ensures the model is never validated on tiles it
# has effectively seen during training.
# =============================================================================

def get_frame_id_from_filename(filename: str) -> str:
    """
    Extract the unique frame ID from a tile filename.
    Used to group all tiles belonging to the same original frame.

    Example:
        "Vid_frame01_tv1_tile5.tif" -> "Vid_frame01"

    Parameters:
        filename (str): Tile filename to extract frame ID from.

    Returns:
        str: Parent frame ID.
    """
    if "_tv" in filename:
        return filename.split("_tv")[0]
    else:
        # Fallback for alternative naming conventions
        return filename.rsplit('_', 2)[0]


def group_files_by_frame(file_paths: List[str]) -> Dict[str, List[str]]:
    """
    Group tile file paths by their parent frame ID.

    Parameters:
        file_paths (List[str]): List of tile image file paths.

    Returns:
        Dict[str, List[str]]: Dictionary mapping frame ID to list of tile paths.
    """
    grouped_files = defaultdict(list)

    for file_path in file_paths:
        filename = os.path.basename(file_path)
        frame_id = get_frame_id_from_filename(filename)
        grouped_files[frame_id].append(file_path)

    return grouped_files


def copy_files_to_split(
    file_list: List[str],
    split_name: str,
    dest_root: str
) -> None:
    """
    Copy image tiles and their corresponding label files to a dataset split folder.

    Creates the following structure:
        dest_root/split_name/images/
        dest_root/split_name/labels/

    Parameters:
        file_list (List[str]): List of image tile paths to copy.
        split_name (str): Name of the split — 'train', 'val', or 'test'.
        dest_root (str): Root directory of the dataset.
    """
    img_dest = os.path.join(dest_root, split_name, "images")
    lbl_dest = os.path.join(dest_root, split_name, "labels")

    os.makedirs(img_dest, exist_ok=True)
    os.makedirs(lbl_dest, exist_ok=True)

    for img_path in tqdm(file_list, desc=f"Populating {split_name}"):
        base_name  = os.path.basename(img_path)
        label_name = base_name.replace(".tif", ".txt")
        label_path = os.path.join(os.path.dirname(img_path), label_name)

        shutil.copy2(img_path, os.path.join(img_dest, base_name))

        if os.path.exists(label_path):
            shutil.copy2(label_path, os.path.join(lbl_dest, label_name))
        # Note: Missing labels are silently skipped — should not occur if
        # Step 2 completed successfully


# --- MAIN EXECUTION ---

# Safety protocol — clears existing dataset to ensure a clean build
if os.path.exists(DATASET_ROOT):
    print(f"Clearing existing dataset directory: {DATASET_ROOT}")
    shutil.rmtree(DATASET_ROOT)

os.makedirs(DATASET_ROOT, exist_ok=True)
print(f"Output directory ready: {DATASET_ROOT}")

# Find all tiled images
all_image_paths = sorted(glob.glob(os.path.join(TILE_OUTPUT_ROOT, "*.tif")))

if not all_image_paths:
    print("ERROR: No TIF images found. Please check TILE_OUTPUT_ROOT in configuration.")
else:
    # Group tiles by parent frame ID to prevent data leakage
    print("Grouping tiles by parent frame ID...")
    grouped_files    = group_files_by_frame(all_image_paths)
    unique_frame_ids = list(grouped_files.keys())

    # Shuffle at frame level — not tile level
    random.seed(RANDOM_SEED)
    random.shuffle(unique_frame_ids)

    # Calculate split sizes
    total_frames = len(unique_frame_ids)
    n_test       = int(total_frames * TEST_RATIO)
    n_val        = int(total_frames * VALIDATION_RATIO)
    n_train      = total_frames - n_test - n_val

    # Assign frames to splits
    train_ids = unique_frame_ids[:n_train]
    val_ids   = unique_frame_ids[n_train:n_train + n_val]
    test_ids  = unique_frame_ids[n_train + n_val:]

    # Flatten frame groups back to file lists
    train_paths = [f for fid in train_ids for f in grouped_files[fid]]
    val_paths   = [f for fid in val_ids   for f in grouped_files[fid]]
    test_paths  = [f for fid in test_ids  for f in grouped_files[fid]]

    print(f"Total unique frames: {total_frames}")
    print(f"Total tiles:         {len(all_image_paths)}")
    print(f"Train: {len(train_paths)} tiles ({len(train_ids)} frames)")
    print(f"Val:   {len(val_paths)} tiles ({len(val_ids)} frames)")
    print(f"Test:  {len(test_paths)} tiles ({len(test_ids)} frames)")

    # Copy files to split directories
    copy_files_to_split(train_paths, "train", DATASET_ROOT)
    copy_files_to_split(val_paths,   "val",   DATASET_ROOT)
    copy_files_to_split(test_paths,  "test",  DATASET_ROOT)

    print(f"Dataset assembly complete.")
    print(f"Saved to: {DATASET_ROOT}")

In [ ]:
# =============================================================================
# STEP 4 — CREATE DATASET YAML
# =============================================================================
# Creates the data.yaml configuration file required by YOLO training.
# Defines dataset paths and class names.
# =============================================================================

import yaml

yaml_content = {
    'path'  : DATASET_ROOT,
    'train' : 'train/images',
    'val'   : 'val/images',
    'test'  : 'test/images',
    'nc'    : 1,
    'names' : {0: 'spot'}
}

yaml_path = '/content/data.yaml'

with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f)

print(f"data.yaml created at: {yaml_path}")

In [ ]:
# =============================================================================
# STEP 5 — MODEL TRAINING
# =============================================================================
# Fine-tunes YOLOv8m on the prepared dataset.
# Augmentation hyperparameters are applied on-the-fly during training.
# Final model weights are saved to MODEL_OUTPUT_DIR in Google Drive.
#
# To resume from a previous checkpoint or use a locally saved base model,
# set BASE_MODEL in the configuration cell to the full path of the .pt file.
# =============================================================================

import random
import numpy as np
import torch

# Set seeds for reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)

model = YOLO(BASE_MODEL)

results = model.train(
    data      = yaml_path,
    epochs    = EPOCHS,
    imgsz     = IMAGE_SIZE,
    batch     = BATCH_SIZE,
    seed      = RANDOM_SEED,
    project   = MODEL_OUTPUT_DIR,
    name      = RUN_NAME,
    cache     = False,
    exist_ok  = True,
    verbose   = True,
    plots     = True,
    # --- On-the-fly augmentation ---
    fliplr    = 0.5,    # Horizontal flip probability
    flipud    = 0.5,    # Vertical flip probability
    degrees   = 25,     # Random rotation range (+/- degrees)
    mosaic    = 1.0,    # Mosaic augmentation probability
    hsv_v     = 0.4     # Brightness/value variation
)

print(f"Training complete. Model saved to: {MODEL_OUTPUT_DIR}")

In [ ]:
# =============================================================================
# STEP 6 — MODEL EVALUATION
# =============================================================================
# Loads the best model weights and evaluates on the held-out test split.
# The test split was never seen during training or validation.
# =============================================================================

best_weights_path = os.path.join(MODEL_OUTPUT_DIR, RUN_NAME, "weights", "best.pt")

best_model   = YOLO(best_weights_path)
test_metrics = best_model.val(split='test')

print(f"Final test mAP50: {test_metrics.box.map50}")